# Airbnb Pricing Model Project

## Problem Statement

### Context

Airbnb is an online platform that allows people to rent short-term accommodation. This ranges from regular people with a spare bedroom to property management firms who lease multiple rentals. On the one side, Airbnb enables owners to list their space and earn rental money. On the other side, it provides travelers easy access to renting private homes.

Airbnb receives commissions from two sources upon every booking, namely from the hosts and guests. For every booking, Airbnb charges the guest 6-12% of the booking fee. Moreover, Airbnb charges the host 3% for every successful transaction.

As a senior data scientist at Airbnb, you have to come up with a pricing model that can effectively predict the Rent for accommodation and can help hosts, travelers, and also the business in devising profitable strategies.

### Objective

To explore and visualize the data, build a linear regression model to predict the prices of Airbnb rental rooms, and generate a set of insights and recommendations that will help the business.

### Data Dictionary

* id: Property ID
  * room_type: Type of Room in the property
  * accommodates: How many adults can this property accommodate
  * bathrooms: Number of bathrooms on the property
  * cancellation_policy: Cancellation policy of the property
  * cleaning_fee: This denotes whether the property cleaning fee is included in the rent or not
  * instant_bookable: It indicates whether an instant booking facility is available or not
  * review_scores_rating: Review rating score of the property
  * bedrooms: Number of bedrooms in the property
  * beds: Total number of beds in the property
  * log_price: Log of the rental price of the property for a fixed period.

# 1\. IMPORT LIBRARIES & LOAD DATA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set visual aesthetics
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Loading dataset...")
# Load the dataset
try:
    df = pd.read_csv('Airbnb.csv')
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Error: 'Airbnb.csv' not found. Please upload the dataset.")
    raise

# 2\. PROBLEM DEFINITION & DATA OVERVIEW (Rubric 1)

In [ ]:
print("\n---2. DATA OVERVIEW ---")
print("""
Questions to answer:
1. What are the primary factors influencing rent?
2. How do room types and cancellation policies affect price?
3. Can we build a reliable linear model to predict these prices?
""")

print("\nData Head:")
print(df.head())

print("\nData Info:")
print(df.info())

print("\nData Description (Numerical):")
print(df.describe())

print("\nData Description (Categorical):")
print(df.describe(include=['O']))

# 3\. EXPLORATORY DATA ANALYSIS (EDA) (Rubric 1 & 2)

**Univariate Analysis**

In [ ]:
print("\n--- 3. EXPLORATORY DATA ANALYSIS ---")

# --- 3.1 Univariate Analysis ---
print("\n[3.1] Univariate Analysis: Distributions & Proportions")

# 1. Target Variable Distribution (Log Price)
plt.figure(figsize=(10, 5))
sns.histplot(df['log_price'], kde=True, color='blue', bins=30)
plt.title('Distribution of Log Price (Target Variable)')
plt.xlabel('Log Price')
plt.show()

# 2. Categorical Proportions (Pie Charts)
fig, ax = plt.subplots(1, 3, figsize=(18, 6))

# Cancellation Policy
df['cancellation_policy'].value_counts().plot.pie(autopct='%1.1f%%', ax=ax[0], shadow=True, startangle=90, cmap='Pastel1')
ax[0].set_title('Cancellation Policy Distribution')
ax[0].set_ylabel('')

# Room Type
df['room_type'].value_counts().plot.pie(autopct='%1.1f%%', ax=ax[1], shadow=True, startangle=90, cmap='Pastel2')
ax[1].set_title('Room Type Distribution')
ax[1].set_ylabel('')

# Instant Bookable
df['instant_bookable'].value_counts().plot.pie(autopct='%1.1f%%', ax=ax[2], shadow=True, startangle=90, cmap='Set3')
ax[2].set_title('Instant Bookable Proportion')
ax[2].set_ylabel('')

plt.show()

# 3. Numerical Distributions (Histograms for key features)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.histplot(df['accommodates'], kde=False, bins=15, color='teal', ax=axes[0,0])
axes[0,0].set_title('Distribution of Accommodates')

sns.histplot(df['review_scores_rating'].dropna(), kde=True, color='purple', bins=20, ax=axes[0,1])
axes[0,1].set_title('Distribution of Review Scores')

sns.countplot(x='bedrooms', data=df, palette='magma', ax=axes[1,0])
axes[1,0].set_title('Count of Bedrooms')

sns.countplot(x='beds', data=df, palette='viridis', ax=axes[1,1])
axes[1,1].set_title('Count of Beds')
plt.tight_layout()
plt.show()

**Observations:**

  1. Log Price: The target variable follows a normal-like distribution centered roughly around 4.5-5.0, which is ideal for Linear Regression assumptions.
  2. Categorical Proportions:
     * Room Type: 'Entire home/apt' constitutes 55.4% and 'Private room' 41.4%, dominating the market. 'Shared rooms' are negligible at 3.0%.
     * Cancellation Policy: 'Strict' is the most common policy (43.7%), followed by 'Flexible' (29.5%) and 'Moderate' (25.7%).
     * Instant Bookable: The majority of listings (73.7%) are NOT instant bookable.
  3. Skewed Features:
     * 'Accommodates' is right-skewed with a clear mode of 2, indicating most listings are for couples.
     * 'Bedrooms' and 'Beds' heavily favor single-bedroom/single-bed units.
     * 'Review Scores' are significantly left-skewed with a massive concentration of scores near 100, indicating a strong ceiling effect or rating inflation.

**Bivariate Analysis**

In [ ]:
# --- 3.2 Bivariate Analysis ---
print("\n[3.2] Bivariate Analysis: Relationships between pairs")

# 1. Violin Plot: Room Type vs Log Price
# (More detailed than boxplot: shows probability density of the data at different values)
plt.figure(figsize=(10, 6))
sns.violinplot(x='room_type', y='log_price', data=df, palette='muted')
plt.title('Density of Log Price by Room Type (Violin Plot)')
plt.show()

# 2. Box Plot: Cleaning Fee vs Log Price
plt.figure(figsize=(8, 5))
sns.boxplot(x='cleaning_fee', y='log_price', data=df, palette='coolwarm')
plt.title('Impact of Cleaning Fee on Price')
plt.xticks([0, 1], ['No Fee (False)', 'Fee Included (True)']) # Adjust labels based on data
plt.show()

# 3. Bar Plot: Average Price by Cancellation Policy
plt.figure(figsize=(10, 6))
avg_price_policy = df.groupby('cancellation_policy')['log_price'].mean().sort_values()
sns.barplot(x=avg_price_policy.index, y=avg_price_policy.values, palette='Blues_d')
plt.title('Average Log Price by Cancellation Policy')
plt.ylabel('Average Log Price')
plt.show()

**Observations:**

  1. Room Type vs Price:
     * 'Entire home/apt' has the highest median log price (approx 5.0) and a much wider distribution, indicating it captures the luxury segment.
     * 'Private room' (median ~4.3) and 'Shared room' (median ~4.0) are significantly cheaper and have tighter price concentrations.
  2. Cleaning Fee:
     * There is a clear separation: Listings with a cleaning fee (True) have a median log_price near 5.0, whereas those without sit closer to 4.5. This suggests higher-value properties almost always charge for cleaning.
  3. Cancellation Policy:
     * A distinct hierarchy exists: 'super_strict_60' and 'super_strict_30' command the highest average prices (exceeding 5.5), representing premium/luxury listings.
     * Standard policies follow a logical order: 'Strict' > 'Moderate' > 'Flexible', confirming that hosts of more expensive properties prefer stricter protections.

**Multivariate Analysis**

In [ ]:
# --- 3.3 Multivariate Analysis ---
print("\n[3.3] Multivariate Analysis: Interactions between 3+ variables")

# 1. Scatter Plot: Accommodates vs Price, Colored by Room Type
# This shows if the relationship between size and price changes depending on the room type
plt.figure(figsize=(12, 7))
sns.scatterplot(x='accommodates', y='log_price', hue='room_type', data=df, alpha=0.7, palette='deep', s=100)
plt.title('Price vs Accommodates (Colored by Room Type)')
plt.show()

# 2. Pairplot
# Visualizes pairwise relationships in a dataset
cols_for_pairplot = ['log_price', 'accommodates', 'bedrooms', 'bathrooms', 'review_scores_rating']
sns.pairplot(df[cols_for_pairplot].dropna(), diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Pairplot of Numerical Variables', y=1.02)
plt.show()

# 3. Correlation Heatmap (Enhanced)
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool)) # Mask upper triangle to avoid duplicates
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Heatmap (Lower Triangle)')
plt.show()

**Observations:**

  1. Key Price Drivers (Correlation):

     * 'accommodates' has the strongest positive correlation with 'log_price' (0.57), followed by 'bedrooms' (0.47) and 'beds' (0.44).
     * 'review_scores_rating' has a surprisingly weak correlation with price (0.09), suggesting that higher ratings increase booking frequency rather than the daily rate itself.
  2. Multicollinearity Alert:

     * There is severe multicollinearity between 'accommodates' and 'beds' (Correlation: 0.81) and 'accommodates' vs 'bedrooms' (0.72).
     * These variables convey very similar information. Using all three might destabilize the model coefficients, though they are all individually good predictors.
  3. Interaction (Scatter Plot):

     * The scatter plot reveals a divergence: For 'Entire home/apt', price increases sharply as capacity grows. For 'Private room', the price curve is much flatter; a 4-person private room isn't much more expensive than a 2-person one, likely due to space constraints.

# 4\. DATA PRE-PROCESSING (Rubric 3)

**4.1 Missing Value Treatment**

In [ ]:
# --- 4.1 Missing Value Treatment ---
print("\nMissing Values before treatment:")
print(df.isnull().sum())

cols_to_impute_median = ['bathrooms', 'bedrooms', 'beds']
for col in cols_to_impute_median:
    df[col].fillna(df[col].median(), inplace=True)

df['review_scores_rating'].fillna(df['review_scores_rating'].mean(), inplace=True)

print("Missing Values after treatment:")
print(df.isnull().sum())

**Observations:**

  1. Major Data Gap (Review Scores): The most significant issue was 'review_scores_rating' with 16,722 missing values. Dropping these rows would have caused massive data loss, justifying the use of mean imputation to preserve the dataset size.
  2. Structural Features: 'Bathrooms' (203), 'Bedrooms' (92), and 'Beds' (131) had moderate missing values. These were imputed with the median to handle integer requirements and skewness.
  3. Minor Gaps: Other columns like 'room_type', 'cancellation_policy', and 'accommodates' had negligible missing values (<10), which have minimal impact on the overall analysis.

**4.2 Feature Engineering & Formatting**

In [ ]:
# --- 4.2 Feature Engineering & Formatting ---
print("\n[4.2] Feature Engineering & Formatting")

if df['cleaning_fee'].dtype == 'object':
    df['cleaning_fee'] = df['cleaning_fee'].map({'TRUE': 1, 'FALSE': 0, 'True': 1, 'False': 0, True: 1, False: 0})

# FIX: Fill any NaNs created by map (if keys didn't match) or existing NaNs with 0
df['cleaning_fee'] = df['cleaning_fee'].fillna(0).astype(int)

# Convert instant_bookable to binary (1/0)
# Usually 't'/'f' in Airbnb data
df['instant_bookable'] = df['instant_bookable'].map({'t': 1, 'f': 0, 'TRUE': 1, 'FALSE': 0})
# FIX: Fill any NaNs created by map with 0
df['instant_bookable'] = df['instant_bookable'].fillna(0).astype(int)

print("Transformed 'cleaning_fee' and 'instant_bookable' to binary numeric format.")
print(df[['cleaning_fee', 'instant_bookable']].head())

**Observations:**

  1. Successful Binary Transformation: The output confirms that 'cleaning_fee' and 'instant_bookable' have been successfully converted to binary integers.
  2. Data Verification: The head of the dataframe shows values of '1' and '0' (e.g., cleaning_fee=1, instant_bookable=0), verifying that the mapping logic handled the original boolean/string data correctly.
  3. Readiness: These features are now mathematically ready for correlation analysis and regression modeling.

**4.3 Outlier Treatment**

In [ ]:
# --- 4.3 Outlier Treatment ---
print("\n[4.3] Outlier Treatment")

# Using IQR method to clip outliers for numerical columns to improve Linear Regression stability
def clip_outliers(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[col] = np.clip(data[col], lower_bound, upper_bound)
    return data

# Apply to continuous variables (excluding log_price as it's the target and already logged)
cols_to_clip = ['accommodates', 'review_scores_rating']
for col in cols_to_clip:
    df = clip_outliers(df, col)

print(f"Applied IQR clipping to: {cols_to_clip}")

**Observations:**

  1. IQR Method: We applied the Interquartile Range (IQR) method to 'accommodates' and 'review_scores_rating'.
  2. Clipping: Instead of removing data points (which loses information), we capped extreme values at the lower and upper bounds (1.5 * IQR). This reduces the leverage of extreme outliers on the regression line while keeping the data points in the analysis.

**4.4 Encoding Categorical Variables**

In [ ]:
# --- 4.4 Encoding Categorical Variables ---
print("\n[4.4] Encoding Categorical Variables")

# Drop 'id' as it is an identifier
df_model = df.drop('id', axis=1)

# One-Hot Encoding for 'room_type' and 'cancellation_policy'
df_encoded = pd.get_dummies(df_model, columns=['room_type', 'cancellation_policy'], drop_first=True)

print("Final Data Shape for Modeling:")
print(df_encoded.shape)
print("Columns:", df_encoded.columns.tolist())

**Observations:**

  1. One-Hot Encoding: Categorical variables 'room_type' and 'cancellation_policy' were transformed into separate binary columns.
  2. Dimensionality: The number of columns increased as each unique category became a new feature.
  3. Drop First: We used 'drop_first=True' (Dummy Variable Trap) to prevent perfect multicollinearity, where one category can be predicted perfectly from the others.

# 5\. MODEL BUILDING - LINEAR REGRESSION (Rubric 4)

In [ ]:
print("\n--- 5. MODEL BUILDING ---")

if df_encoded.isnull().sum().sum() > 0:
    print(f"Warning: Found {df_encoded.isnull().sum().sum()} missing values. Filling with 0.")
    df_encoded.fillna(0, inplace=True)

# Define X (Features) and y (Target)
X = df_encoded.drop('log_price', axis=1)
y = df_encoded['log_price']

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and Fit Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Model Coefficients
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr_model.coef_})
coef_df = coef_df.sort_values(by='Coefficient', ascending=False)

print("\nTop 5 Positive Influencers on Price:")
print(coef_df.head(5))

print("\nTop 5 Negative Influencers on Price:")
print(coef_df.tail(5))

**Observations:**

  1. Physicality is Key (Positive Drivers):

     * 'bedrooms' (0.155), 'bathrooms' (0.147), and 'accommodates' (0.100) are the top positive predictors.
     * Interestingly, only these 3-4 features strongly drive price up. 'review_scores_rating' is positive but has a very small impact (0.010).
  2. The Cost of Privacy (Negative Drivers):

     * 'room_type_Shared room' (-1.07) and 'room_type_Private room' (-0.61) are the strongest negative coefficients.
     * Compared to the baseline ('Entire home/apt'), renting a shared room drops the log-price by over 1.0, which is a massive devaluation.
  3. Multivariate vs. Bivariate (The "Surprise"):

     * 'cleaning_fee' (-0.075) and 'cancellation_policy_strict' (-0.012) have negative coefficients, contradicting our EDA where they seemed to have higher prices.
     * Insight: High prices are driven by the property size (bedrooms/bathrooms). Once we control for size, adding a cleaning fee or a strict policy actually correlates with a slightly lower base rent. This suggests that the "premium" we saw in EDA was purely due to the fact that bigger homes happen to have fees/strict policies, not because the policies themselves generate value.

# 6\. MODEL PERFORMANCE EVALUATION (Rubric 5)

In [ ]:
print("\n--- 6. MODEL PERFORMANCE EVALUATION ---")

# Predictions
y_pred_train = lr_model.predict(X_train)
y_pred_test = lr_model.predict(X_test)

# Metrics
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Training R-squared: {train_r2:.4f}")
print(f"Testing R-squared:  {test_r2:.4f}")
print(f"Testing RMSE:       {test_rmse:.4f}")
print(f"Testing MAE:        {test_mae:.4f}")

# Residual Plot to check for Homoscedasticity
residuals = y_test - y_pred_test
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_pred_test, y=residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Log Price')
plt.ylabel('Residuals')
plt.title('Residual Plot (Homoscedasticity Check)')
plt.show()

# Actual vs Predicted Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred_test, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Log Price')
plt.ylabel('Predicted Log Price')
plt.title('Actual vs Predicted Log Price')
plt.show()

**Observations:**

  1. Physicality is Key (Positive Drivers):

     * 'bedrooms' (0.155), 'bathrooms' (0.147), and 'accommodates' (0.100) are the top positive predictors.
     * Interestingly, only these 3-4 features strongly drive price up. 'review_scores_rating' is positive but has a very small impact (0.010).
  2. The Cost of Privacy (Negative Drivers):

     * 'room_type_Shared room' (-1.07) and 'room_type_Private room' (-0.61) are the strongest negative coefficients.
     * Compared to the baseline ('Entire home/apt'), renting a shared room drops the log-price by over 1.0, which is a massive devaluation.
  3. Multivariate vs. Bivariate (The "Surprise"):

     * 'cleaning_fee' (-0.075) and 'cancellation_policy_strict' (-0.012) have negative coefficients, contradicting our EDA where they seemed to have higher prices.
     * Insight: High prices are driven by the property size (bedrooms/bathrooms). Once we control for size, adding a cleaning fee or a strict policy actually correlates with a slightly lower base rent. This suggests that the "premium" we saw in EDA was purely due to the fact that bigger homes happen to have fees/strict policies, not because the policies themselves generate value.

# 7\. ACTIONABLE INSIGHTS & RECOMMENDATIONS (Rubric 6)

**Based on the comprehensive EDA and Linear Regression analysis, here are the final strategic recommendations:**

  1. **Investment Strategy: Maximize Capacity & Bathrooms**

     * **Insight:** 'Bedrooms' (+0.155), 'Bathrooms' (+0.147), and 'Accommodates' (+0.100) are the strongest positive levers.
     * **Action:** For property owners, adding a second bathroom or optimizing layout for an extra bedroom yields the highest ROI. A 2-bed/2-bath unit commands a significantly higher premium than a 2-bed/1-bath.
  2. **Room Type Positioning: Avoid "Shared" Listings**

     * **Insight:** The "Shared room" label carries a massive price penalty (coefficient -1.07), and "Private room" is -0.61 relative to entire homes.
     * **Action:** If a property allows, prioritize converting space into self-contained "Entire home/apt" units (even small studios). If offering a private room, ensure it's distinct enough to avoid the "shared" perception.
  3. **Pricing Strategy: The "Premium" Fallacy (Cleaning Fees & Instant Book)**

     * **Insight:** While expensive homes often have cleaning fees, the model shows that _holding size constant_ , adding a cleaning fee (-0.075) or enabling Instant Book (-0.062) correlates with lower daily rates.
     * **Action:** \- **Cleaning Fees:** Do not use cleaning fees to hide rent. High fees may deter bookings without adding perceived value. Use them strictly for cost recovery.
       * **Instant Book:** Use this feature to drive volume (occupancy), not price. It attracts last-minute travelers looking for deals/convenience, not luxury.
  4. **Platform Strategy: The Data Gap**

     * **Insight:** The model explains ~50% of price variance. The remaining 50% is almost certainly driven by **Location** (which is missing).
     * **Action:** To build a "Gold Standard" pricing tool, Airbnb must integrate granular geospatial data (neighborhood desirability scores, proximity to transit). Without this, price predictions will have a +/- 40% margin of error.
  5. **Reputation Management: The 90+ Threshold**

     * **Insight:** Review scores have a weak linear correlation with price (+0.01), but the distribution is heavily skewed to 100.
     * **Action:** A score of 95+ is "table stakes" to remain competitive, but moving from 95 to 100 won't justify a massive price hike. Scores are a conversion tool, not a pricing lever. Drop below 90, and visibility/volume will likely crash.